In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from db import query

In [ ]:
# Load meta-features
meta = pd.read_csv('results/metafeature_descriptors.csv')

# Load bonferroni labels
bonf = query("SELECT feature, feature_group, bonferroni_rejected FROM bonferroni_fdr_results")

# Merge on feature name
df = meta.merge(bonf, on='feature', how='inner')

# Label: bonferroni_rejected = 1 (spurious), 0 (genuine)
df['label'] = df['bonferroni_rejected'].astype(int)

print(f"Total features: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"By group:\n{df.groupby(['feature_group_x','label']).size()}")

In [ ]:
# One-hot encode feature group
group_dummies = pd.get_dummies(df['feature_group_x'], prefix='group', drop_first=True)

X = pd.concat([
    df[['corr_stability']].fillna(0),
    group_dummies,
], axis=1)

y = df['label'].values

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {list(X.columns)}")
print(f"Class balance: {y.mean():.3f} positive rate")

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_scores = []
fold_aucs = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_scaled, y)):
    clf = LogisticRegression(penalty='l1', solver='liblinear', C=1.0, random_state=42)
    clf.fit(X_scaled[train_idx], y[train_idx])

    preds = clf.predict(X_scaled[val_idx])
    probs = clf.predict_proba(X_scaled[val_idx])[:, 1]

    acc = (preds == y[val_idx]).mean()
    auc = roc_auc_score(y[val_idx], probs)
    fold_scores.append(acc)
    fold_aucs.append(auc)
    print(f"Fold {fold+1}: acc={acc:.3f}, auc={auc:.3f}")

print(f"\nCV Accuracy: {np.mean(fold_scores):.3f} \u00b1 {np.std(fold_scores):.3f}")
print(f"CV AUC:      {np.mean(fold_aucs):.3f} \u00b1 {np.std(fold_aucs):.3f}")

In [ ]:
clf_final = LogisticRegression(penalty='l1', solver='liblinear', C=1.0, random_state=42)
clf_final.fit(X_scaled, y)

coef_df = pd.DataFrame({
    'feature': list(X.columns),
    'coefficient': clf_final.coef_[0]
}).sort_values('coefficient', ascending=False)

print(coef_df)

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(coef_df['feature'], coef_df['coefficient'], edgecolor='black')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Meta-Feature Logistic Regression Coefficients (L1)')
ax.set_xlabel('Coefficient')
plt.tight_layout()
plt.savefig('plots/meta_regression_coefficients.png', dpi=150)
plt.show()

In [ ]:
text_df = df[df['feature_group_x'] == 'text'].copy()

text_group_dummies = pd.get_dummies(text_df['feature_group_x'], prefix='group', drop_first=True)
# Align columns to training matrix
text_group_dummies = text_group_dummies.reindex(columns=group_dummies.columns, fill_value=0)

X_text = pd.concat([
    text_df[['corr_stability']].fillna(0).reset_index(drop=True),
    text_group_dummies.reset_index(drop=True),
], axis=1)

X_text_scaled = scaler.transform(X_text)
text_df = text_df.copy()
text_df['predicted_prob'] = clf_final.predict_proba(X_text_scaled)[:, 1]
text_df['predicted_label'] = clf_final.predict(X_text_scaled)

# Breakdown by text subgroup
def subgroup(feat):
    if feat == 'sentiment_score': return 'sentiment'
    if feat.startswith('lda_topic'): return 'lda'
    return 'embeddings'

text_df['subgroup'] = text_df['feature'].apply(subgroup)

print(text_df.groupby('subgroup').agg(
    n=('feature','count'),
    predicted_spurious=('predicted_label','sum'),
    avg_prob=('predicted_prob','mean')
).round(3))

In [ ]:
labeled = df[df['feature_group_x'].isin(['control','spurious'])].copy()

labeled_group_dummies = pd.get_dummies(labeled['feature_group_x'], prefix='group', drop_first=True)
labeled_group_dummies = labeled_group_dummies.reindex(columns=group_dummies.columns, fill_value=0)

X_labeled = pd.concat([
    labeled[['corr_stability']].fillna(0).reset_index(drop=True),
    labeled_group_dummies.reset_index(drop=True),
], axis=1)

X_labeled_scaled = scaler.transform(X_labeled)
preds = clf_final.predict(X_labeled_scaled)

print(classification_report(labeled['label'].values, preds,
      target_names=['genuine','spurious']))